In [627]:
import time
import random
import string
from collections import defaultdict

In [628]:
class Contract:
    def __init__(self):
        self.address = '0x' + ''.join(random.choices('0123456789abcdef', k=40))
    def emit(self, topic, *args): print(f"{topic} - {' '.join(map(str, args))}")

In [727]:
class AddressDict(defaultdict):
    def __getitem__(self, key):
        return super().__getitem__(key.address if hasattr(key, 'address') else key)

    def __setitem__(self, key, value):
        return super().__setitem__(key.address if hasattr(key, 'address') else key, value)

In [728]:
class User(Addressable):
    def __str__(self): return f"{self.address[:5]}"

In [729]:
shafu   = User()
wolfram = User()
taleb   = User()
deutsch = User()
popper  = User()

## ERC-20

https://github.com/transmissions11/solmate/blob/main/src/tokens/ERC20.sol

In [730]:
ZERO_ADDRESS = "0x0"

In [734]:
class Token(Contract):
    def __init__(self, name, symbol): # no need for decimals
        super().__init__()
        self.name        = name
        self.symbol      = symbol
        self.totalSupply = 0
        self.allowance   = AddressDict(lambda: AddressDict(int))
        self.balanceOf   = AddressDict(int)

    def approve(self, sender, spender, amount):
        self.allowance[sender][spender] = amount;
        self.emit("Approval", sender, spender, amount)
        return True

    def transfer(self, sender, to, amount):
        assert self.balanceOf[sender] >= amount
        self.balanceOf[sender] -= amount
        self.balanceOf[to]     += amount
        self.emit("Transfer", sender, to, amount)
        return True        

    def transferFrom(self, fr, sender, to, amount):
        assert self.allowance[fr][sender] >= amount
        self.allowance[fr][sender] -= amount
        self.balanceOf[sender]     -= amount
        self.balanceOf[to]         += amount
        self.emit("Transfer", sender, to, amount)
        return True

    def mint(self, to, amount):
        self.balanceOf[to] += amount
        self.totalSupply   += amount
        self.emit("Transfer", ZERO_ADDRESS, to, amount)

    def burn(self, fr, amount):
        self.balanceOf[fr.address]  = 0
        self.balanceOf[fr.address] -= amount
        self.totalSupply           -= amount

In [735]:
DYAD = Token("DYAD Stable",   "DYAD")
USDC = Token("USD Coin",      "USDC")
WETH = Token("Wrapped Ether", "WETH")
WBTC = Token("Wrapped BTC",   "WBTC")
USDT = Token("Tether USD",    "USDT")

In [736]:
DYAD.mint    (wolfram, 100)

Transfer - 0x0 0x851 100


In [608]:
class Oracle(Contract):
    def __init__(self, asset0, asset1, price): 
        self.asset0, self.asset1 = asset0, asset1
        self.price = price
    def set_price(self, price): self.price = price
    def __str__(self): 
        return f"{self.asset0.symbol}/{self.asset1.symbol} {self.price}"

In [737]:
ETH2USD = Oracle(WETH, USDC, 2615.93)
ETH2BTC = Oracle(WETH, WBTC, 0.039)

print(ETH2USD)
print(ETH2BTC)

WETH/USDC 2615.93
WETH/WBTC 0.039


### utils

In [343]:
def get_time():    return int(time.time())
def apr_to_spr(x): return x / (365*24*60*60) # interest / second

In [358]:
INTEREST_PER_YEAR = 0.02  # 2%
INTEREST_SLOPE    = 0.056
RESERVE_FACTOR    = 0.9   # 90%

In [406]:
class MicroLend(Contract):
    def __init__(
        self,
        lend_asset,
        collat_asset,
        borrow_asset,
        min_collat_ratio
    ):
        self.lend_asset        = lend_asset
        self.collat_asset      = collat_asset
        self.borrow_asset      = borrow_asset
        self.min_collat_ratio  = min_collat_ratio
        self.balances          = defaultdict(lambda: defaultdict(int))
        self.totals            = defaultdict(int)
        
        self.reward_per_token      = 0
        self.reward_per_token_paid = defaultdict(int)
        self.rewards               = defaultdict(int)
        self.updatedAt             = get_time()

    def get_reward_per_token(self):
        totalLend = self.totals[self.lend_asset.address]
        if totalLend == 0: return self.reward_per_token

        spr = apr_to_spr(INTEREST_PER_YEAR)   # interest / second
        dt  = get_time() - self.updatedAt

        return self.reward_per_token + (spr * dt) / totalLend

    def earned(self, user):
        balance = self.balances[user.address][self.lend_asset.address]
        d_rewardPerToken = self.reward_per_token - self.reward_per_token_paid[user.address]
        return self.rewards[user.address] + (balance * d_rewardPerToken)

    def updateReward(self, user):
        self.reward_per_token = self.get_reward_per_token()
        self.updatedAt        = get_time()

        self.rewards[user.address]               = self.earned(user)
        self.reward_per_token_paid[user.address] = self.reward_per_token

    def deposit(self, user, token, amount):
        self.updateReward(user)
        self.balances[user.address][token.address] += amount
        self.totals[token.address]                 += amount

    def withdraw(self, user, token, amount):
        self.balances[user.address][token.address] -= amount
        self.totals[token.address]                 -= amount

    def borrow(self, user, amount):
        self.balances[user.address][self.borrow_asset.address] += amount
        self.totals[self.borrow_asset.address]                 += amount

    def repay(self, user, amount): pass

    def collat_ratio(self, user): 
        borrowed = self.balances[user.address][self.borrow_asset.address]
        if borrowed == 0: return (1<<256) - 1 # a very large uint 
        return self.balances[user.address][self.collat_asset.address] / borrowed

    def utilization(self):
        lend = self.totals[self.lend_asset.address]
        if lend == 0: return 0
        return self.totals[self.borrow_asset.address] / lend

    def borrow_apr(self): return self.utilization() * INTEREST_SLOPE
    def lend_apr  (self): return self.borrow_apr()  * RESERVE_FACTOR

    def stats(self, user):
        mL.updateReward(user)
        lend   = self.balances[user.address][self.lend_asset.address]
        collat = self.balances[user.address][self.collat_asset.address]
        borrow = self.balances[user.address][self.borrow_asset.address]
        cr     = round(self.collat_ratio(user)*100, 2)
        reward = self.rewards[user.address]
        print(
            f"{'Lend:':<14} {lend} {self.lend_asset.symbol}\n"
            f"{'Collateral:':<14} {collat}\n"
            f"{'Borrowed:':<14} {borrow} {self.borrow_asset.symbol}\n"
            f"{'Collat Ratio:':<14} {cr}%\n"
            f"{'Reward:':<14} {reward}%\n"
            "\n"
            f"{'Utilization:':<14} {self.utilization()*100}%\n"
            f"{'Lend APR:':<14} {self.lend_apr()*100}%\n"
            f"{'Borrow APR:':<14} {self.borrow_apr()*100}%"
        )

In [413]:
mL = MicroLend(
    WETH, 
    USDC,
    DYAD,
    1.5   # 150%
)

In [414]:
mL.deposit(vitalik, WETH, 1_000)
mL.borrow (vitalik, 130)

In [439]:
mL.stats(vitalik)

Lend:          1000 WETH
Collateral:    200
Borrowed:      130 DYAD
Collat Ratio:  153.85%
Reward:        8.561643835616443e-08%

Utilization:   13.0%
Lend APR:      0.6552%
Borrow APR:    0.728%
